# D11. Представление весов и поиск кодов

После D10 осталось два объяснения проигрыша. Может не хватать уровней выбранной сетки, а может плохо работать поиск по уровням, которые уже есть. Заморозил формат и дал тем же рецептам более сильный поиск.

**Результат.** Локальная ошибка на калибровке снизилась на **42–67%**. На OPT отражаемая сетка выиграла у GPTQ на одном подмножестве. Полную серию из трёх подмножеств и ближайших контролей проверил [в D12](../D12/experiments.ipynb).

[Назад: форматы](../D10/experiments.ipynb) · [Протокол дня](README.md) · [Дальше: диагностика и бюджет](../D12/experiments.ipynb)

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

day = Path("quantization/research/D11")
for directory in (Path.cwd(), *Path.cwd().parents):
    if (directory / day / "results").is_dir():
        results = directory / day / "results"
        break
else:
    raise FileNotFoundError(f"Не найдены CSV в {day}/results")

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 120})

## Что меняется при full-H

Если ошибка строки весов равна $e=\hat w-w$, то квадратичная ошибка выхода пропорциональна $e^\mathsf{T}He$, где $H$ — матрица вторых моментов входных активаций. Диагональ учитывает амплитуды отдельных признаков. Внедиагональные члены позволяют одному изменению кода компенсировать ошибку другого.

Ниже маленькая иллюстрация одного координатного прохода. При изменении координаты $i$ на $\delta$ энергия меняется на $2\delta(He)_i+\delta^2 H_{ii}$. В полном эксперименте E0 стартует с сохранённых кодов D10, делает два прохода по кодам, условную подгонку таблиц и ещё один проход. Панель включает 288 кодирований: две модели, три подмножества, 12 матриц и четыре формата.

In [2]:
def coordinate_pass(weights, levels, hessian):
    codes = levels[np.abs(weights[:, None] - levels).argmin(axis=1)].copy()
    error = codes - weights
    for index in range(len(weights)):
        delta = levels - codes[index]
        loss_change = 2 * delta * (hessian @ error)[index] + delta**2 * hessian[index, index]
        codes[index] = levels[loss_change.argmin()]
        error[index] = codes[index] - weights[index]
    return codes


weights = np.array([.43, .42])
levels = np.array([0., 1.])
hessian = np.array([[1., .9], [.9, 1.]])
diagonal_codes = coordinate_pass(weights, levels, np.diag(np.diag(hessian)))
full_codes = coordinate_pass(weights, levels, hessian)
for label, codes in [("Диагональ", diagonal_codes), ("Полная H", full_codes)]:
    error = codes - weights
    print(f"{label}: коды {codes.tolist()}, энергия под полной H {error @ hessian @ error:.4f}")

Диагональ: коды [0.0, 0.0], энергия под полной H 0.6864
Полная H: коды [1.0, 0.0], энергия под полной H 0.0704


В рабочих прогонах основной выигрыш дал пересмотр кодов с полной H. Подгонка таблиц обычно добавляла ещё **0,06–0,21%**. Разрыв между представлениями на панели сохранился.

## PPL полной модели на subset 0

Для OPT здесь три оценки. Для SmolLM2 сравниваю две E0-конфигурации из отдельной локальной очереди. Сдвиг в процентах считаю внутри одной модели и одного подмножества.

In [3]:
runs = pd.read_csv(results / "encoder_ppl.csv")
assert runs["subset"].eq(0).all()
display(runs.sort_values(["model", "ppl"]).round({"ppl": 6}))
opt = runs[runs["model"].eq("OPT-125M")].set_index("method")["ppl"]
gain = 100 * (1 - opt["window_mirror"] / opt["GPTQ_native"])
print(f"OPT, subset 0: снижение PPL к GPTQ {gain:.3f}%")

,model,method,subset,ppl,targets
1,OPT-125M,window_mirror,0,28.027490,247288
4,OPT-125M,GPTQ_native,0,28.267954,247288
0,OPT-125M,trellis_full,0,28.283723,247288
2,SmolLM2-135M,trellis_full,0,16.952380,265681
3,SmolLM2-135M,window_mirror,0,16.994216,265681


OPT, subset 0: снижение PPL к GPTQ 0.851%


## Два локальных теста

| Вариант | Контроль | Что получилось |
| --- | --- | --- |
| H1, взаимодействие трёх 4-битных кодов через общую таблицу | Более свободная таблица из той же стартовой точки | 72 кодирования на 36 матрицах. Отбор по калибровке не прошёл на обеих моделях. Full-model PPL для H1 не измерялась. |
| Смешивание каналов в gated MLP | Identity с учётом дрейфа FP-выхода | На девяти участках ошибка на резервной калибровке в среднем примерно на 1,9% выше identity. |

Качество формата зависит от того, насколько хорошо подбираются коды. Локальную ошибку использую как диагностику, а полную PPL — для вывода о модели.

## Таблицы и исходные записи

Слева — CSV, которые читает ноутбук. Справа — имена исходных локальных записей.

- [encoder_ppl.csv](results/encoder_ppl.csv) · `private/attempt4/downloaded_metadata/results/jobs/OPT__trellis_full__s0/report.json`, `private/attempt4/downloaded_metadata/results/jobs/OPT__window_mirror__s0/report.json`, `private/local_gate1/results/full_Smol_trellis_full_s0.json`, `private/local_gate1/results/full_Smol_window_mirror_s0.json`, `private/ИТОГ.md`